# Probabilistic ReAct trajectories, with candidate probabilities

Same agent configuration as `traceletReAct.ipynb` (cells 1-5 are copied from
`traceletReAct_single.ipynb`), but built for reading one trajectory rather than sweeping.

For each step you see the sampled candidates side by side -- their fill-in probability, the
judge's score out of 10, the code, the resulting observation -- with the winner starred, followed
by the code that was actually committed.

`N_SAMPLES = 3` by default; at `N_SAMPLES = 1` there is no judge and nothing to compare.

Two ways to use it:
* **live** -- run the `runone` cell, then `show`.
* **saved** -- skip to `from_pickle` and point it at any `{i}.pkl` from a completed sweep.

Fill-in probabilities need a model that reports logprobs, so gpt-4o, gpt-5.4-mini and
Qwen3.5-9B have them and Qwen3.7-Plus does not (it is streaming-only and its upstream
refuses `n>1` with `logprobs`). Runs made before per-candidate recording still show the
probabilities, just without each candidate's code and judge score.


In [44]:
# Pick up edits to the editable-installed smolagents without a kernel restart.
%load_ext autoreload
%autoreload 2

# ===================== experiment configuration =====================
MODEL = "gpt-5.4-mini"                 # key of MODELS below
SKELETON_STRATEGY = "direct_prompt"     # "direct_prompt" | "post_process"
N_SAMPLES = 3                          # candidates per step; needs >1 for probabilities to mean anything
USE_TRACELET_PROMPT = True             # tracelet_agent.yaml (template/fill-in protocol) vs default code_agent.yaml
QUESTION_INDEX = 4                 # single GAIA validation index to run
N_QUESTIONS = None                   # unused here; kept so the config block matches the sweep notebook
MAX_STEPS = 50
RUN_TAG = "_v5"                   # e.g. "_rerun" for fresh output paths
# ====================================================================

MODELS = {
    "gpt-4o": "openai",
    "gpt-5.4-mini": "openai",
    "Qwen/Qwen3.5-9B": "together",
    "Qwen/Qwen3.7-Plus": "together",
}
ENABLE_THINKING = False  # Together-served open-weight models only

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; pick one of {list(MODELS)}"
assert SKELETON_STRATEGY in ("direct_prompt", "post_process")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = MODEL
provider = MODELS[MODEL]

# Only Qwen3.7-Plus rejects non-streaming, and streaming drops fill-in logprobs, so stream only when forced.
STREAM_ONLY = {"Qwen/Qwen3.7-Plus"}

if provider == "openai":
    model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
else:
    # Together needs enable_thinking nested in chat_template_kwargs for vLLM-served models (see naiveReAct).
    model = OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
        client_kwargs={"timeout": 300.0},  # bound a stalled stream instead of hanging forever
    )
stream_outputs = model_name in STREAM_ONLY

print(f"{model_name} via {provider} | stream_outputs={stream_outputs}")

gpt-5.4-mini via openai | stream_outputs=False


In [46]:
# Standard GAIA tool stack, same as naiveReAct.ipynb.
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

web_search: LeakFilteredDuckDuckGoSearchTool at 1.0 q/s


In [47]:
import importlib.resources

import yaml

from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

# None lets CodeAgent fall back to its default code_agent.yaml.
prompt_templates = None
if USE_TRACELET_PROMPT:
    prompt_templates = yaml.safe_load(
        importlib.resources.files("smolagents.prompts").joinpath("tracelet_agent.yaml").read_text()
    )

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=MAX_STEPS,
    verbosity_level=LogLevel.DEBUG,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=N_SAMPLES,
    skeleton_strategy=SKELETON_STRATEGY,
    judge_strategy="score_probs",   # "scores" | "score_probs" | "token_probs"
    stream_outputs=stream_outputs,
    prompt_templates=prompt_templates,
)

In [48]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [49]:
# Trajectory renderer. Works on a live agent's memory or on a saved {i}.pkl from any sweep.
import html
import math
import pickle
from pathlib import Path

from IPython.display import HTML, display

from smolagents.memory import ActionStep

# Raise these if a step's observation is being cut off where you need to read it.
OBS_LIMIT = 4000        # committed observation, per step
CAND_OBS_LIMIT = 900    # per candidate (three of them per step, so kept tighter)
CODE_LIMIT = 800


def step_records(source):
    """Normalise a live agent, an ActionStep list, or a run pickle into plain step dicts."""
    if isinstance(source, (str, Path)):
        with open(source, "rb") as f:
            return pickle.load(f)["steps"]
    steps = source.memory.steps if hasattr(source, "memory") else source
    # A list of plain dicts is already normalised (e.g. steps pulled straight out of a pickle).
    if steps and all(isinstance(s, dict) for s in steps):
        return list(steps)
    out = []
    for s in steps:
        if not isinstance(s, ActionStep):
            continue
        out.append({
            "step_number": s.step_number,
            "model_output": s.model_output,
            "code_action": s.code_action,
            "observations": s.observations,
            "error": str(s.error) if s.error else None,
            "is_final_answer": s.is_final_answer,
            "token_usage": s.token_usage.dict() if s.token_usage else None,
            "sentinel_count": s.sentinel_count,
            "fillin_logprobs": s.fillin_logprobs,
            "candidates": s.candidates,
        })
    return out


def _bar(value, vmax, colour):
    if value is None:
        return "&ndash;"
    width = 0 if not vmax else max(2, round(100 * value / vmax))
    return (f"<div style='background:{colour};height:9px;width:{width}px;display:inline-block;"
            f"vertical-align:middle'></div> {value:.3f}")


def _block(label, text, limit, colour="#bbb"):
    """Labeled text block. Always shows the true size, so a clipped observation is obvious."""
    text = (text or "").strip()
    if not text:
        return f"<div style='font-size:11px;color:#999;margin-top:6px'>{label}: (empty)</div>"
    shown = text[:limit]
    hidden = len(text) - len(shown)
    tail = f"\n\n… {hidden:,} more characters (raise OBS_LIMIT to see them)" if hidden else ""
    return (f"<div style='font-size:11px;color:#666;margin-top:6px'>{label}"
            f" <span style='color:#999'>&middot; {len(text):,} chars</span></div>"
            f"<pre style='margin:2px 0;white-space:pre-wrap;font-size:11px;"
            f"border-left:3px solid {colour};padding-left:8px'>"
            f"{html.escape(shown + tail)}</pre>")


def _candidate_table(cands, raw=False):
    # "scores" judges are 0-10; score_probs/token_probs are probabilities summing to 1.
    kind = (cands[0].get("judge_kind") or "scores") if cands else "scores"
    jmax, jlabel = (10.0, "judge /10") if kind == "scores" else (1.0, f"judge p ({kind})")
    cols = ["cand", "fill-in p", jlabel, ""]
    if raw:
        cols = ["cand", "logprob", "P(fill-in)", "tok", "mean logprob", "P per token", jlabel, ""]
    rows = []
    for i, c in enumerate(cands):
        won = c.get("won")
        mark = "&#9733;" if won else ""
        edge = "3px solid #2e7d32" if won else "3px solid transparent"
        weight = "600" if won else "400"
        judge = c.get("judge_score")
        state = "error" if c.get("error") else ("final answer" if c.get("is_final_answer") else "")
        td = "padding:3px 8px"
        cells = [f"<td style='{td}'>{mark} c{i}</td>"]
        if raw:
            lp, ntok, mlp = c.get("fillin_logprob"), c.get("fillin_n_tokens"), c.get("fillin_mean_logprob")
            # exp() of the summed logprob is the model's own probability for that whole fill-in.
            cells += [
                f"<td style='{td}'>{'&ndash;' if lp is None else format(lp, '+.4f')}</td>",
                f"<td style='{td}'>{'&ndash;' if lp is None else format(math.exp(lp), '.6g')}</td>",
                f"<td style='{td}'>{'&ndash;' if ntok is None else ntok}</td>",
                f"<td style='{td}'>{'&ndash;' if mlp is None else format(mlp, '+.4f')}</td>",
                f"<td style='{td}'>{'&ndash;' if mlp is None else format(math.exp(mlp), '.6f')}</td>",
                f"<td style='{td}'>{'&ndash;' if judge is None else format(judge, '.4f' if jmax == 1.0 else '.1f')}</td>",
            ]
        else:
            cells += [
                f"<td style='{td}'>{_bar(c.get('fillin_prob'), 1.0, '#5c6bc0')}</td>",
                f"<td style='{td}'>{_bar(judge, jmax, '#26a69a')}</td>",
            ]
        cells.append(f"<td style='{td};color:#c62828'>{state}</td>")
        rows.append(
            f"<tr style='border-left:{edge};font-weight:{weight}'>{''.join(cells)}</tr>"
            f"<tr><td></td><td colspan='{len(cols) - 1}' style='padding:0 8px 10px'>"
            f"{_block('code', c.get('code'), CODE_LIMIT, '#5c6bc0')}"
            f"{_block('observation', c.get('observation'), CAND_OBS_LIMIT, '#90a4ae')}"
            f"</td></tr>"
        )
    header = "".join(f"<th style='padding:3px 8px'>{h}</th>" for h in cols)
    return (f"<table style='border-collapse:collapse;width:100%'>"
            f"<tr style='text-align:left;font-size:11px;color:#666'>{header}</tr>{''.join(rows)}</table>")


def render_trajectory(source, question=None, true_answer=None, prediction=None, max_steps=None,
                      raw=False):
    """Show every step: the candidates with their fill-in probabilities, the judge's scores,
    which one won, and then the code and observation actually committed.

    raw=True replaces the probability bars with the model's own numbers: the summed sequence
    logprob and its exp() -- the actual probability the model assigned that whole fill-in -- plus
    the token count and the per-token mean and its exp(). These are absolute likelihoods, not
    normalised across candidates, so they do not sum to 1."""
    records = step_records(source)
    if max_steps:
        records = records[:max_steps]
    parts = []
    if question:
        parts.append(f"<div style='font-size:12px'><b>Q</b> {html.escape(str(question))}</div>")
    if true_answer is not None or prediction is not None:
        parts.append(f"<div style='font-size:12px'><b>true</b> {html.escape(str(true_answer))}"
                     f" &nbsp;|&nbsp; <b>predicted</b> {html.escape(str(prediction))}</div>")

    for r in records:
        cands = r.get("candidates")
        lps = r.get("fillin_logprobs")
        tok = (r.get("token_usage") or {}).get("total_tokens")
        head = f"Step {r.get('step_number')}"
        if tok:
            head += f" &middot; {tok:,} tok"
        if cands:
            head += f" &middot; {len(cands)} candidates"
        elif r.get("sentinel_count") == 0:
            head += " &middot; no sentinel (direct execution)"
        parts.append(f"<div style='margin-top:20px;padding-top:6px;border-top:1px solid #ddd;"
                     f"font-weight:600;font-size:12px'>{head}</div>")

        if r.get("model_output"):
            parts.append("<details><summary style='font-size:11px;color:#666;cursor:pointer'>"
                         "model output (thought + skeleton)</summary>"
                         f"{_block('', r['model_output'], 2500)}</details>")

        if cands:
            parts.append(_candidate_table(cands, raw=raw))
        elif lps:
            # Runs from before per-candidate recording still carry the probabilities.
            if raw:
                probs = " | ".join(
                    f"c{i}: logprob={e.get('logprob', float('nan')):+.4f}"
                    f" P={math.exp(e['logprob']):.6g}" if e.get("logprob") is not None else f"c{i}: &ndash;"
                    for i, e in enumerate(lps)
                )
            else:
                probs = "  ".join(f"c{i}={e.get('prob', float('nan')):.3f}" for i, e in enumerate(lps))
            parts.append(f"<div style='font-size:11px;color:#666'>fill-in probs: {probs} "
                         f"(per-candidate code/judge not recorded in this run)</div>")

        parts.append(_block("COMMITTED CODE", r.get("code_action"), CODE_LIMIT, "#2e7d32"))
        if r.get("error"):
            parts.append(f"<div style='font-size:11px;color:#c62828;margin-top:6px'>error: "
                         f"{html.escape(r['error'][:400])}</div>")
        parts.append(_block("COMMITTED OBSERVATION", r.get("observations"), OBS_LIMIT, "#2e7d32"))

    path_rows, totals = _path_totals(records)
    parts.append("<div style='margin-top:22px;font-weight:600;font-size:12px'>"
                 "Path probability (winning candidate at each step)</div>")
    parts.append(_path_html(path_rows, totals))
    display(HTML(f"<div style='font-family:ui-monospace,monospace'>{''.join(parts)}</div>"))
    return totals


def _path_totals(records):
    """Multiply the winning candidate's probabilities along the committed path.

    Three quantities, kept separate because they answer different questions:
      * P(fill-ins)      product of the model's own likelihoods for the fill-ins actually used.
                         A joint density over more tokens each step, so it shrinks with trace
                         length and is only comparable between paths of similar length.
      * selection weight product of the winner's normalised fill-in probability. Compare it to
                         the uniform baseline: equal means the mechanism was indifferent.
      * judge weight     same for the judge's probability, when the run used score_probs or
                         token_probs. The 0-10 `scores` judge is not a probability, so it is
                         skipped rather than silently multiplied.
    Steps with no sentinel sampled nothing and contribute no factor.
    """
    rows = []
    logp = log_sel = log_judge = log_unif = 0.0
    n_judge = n_missing = 0
    for r in records:
        cands = r.get("candidates")
        if not cands:
            if r.get("fillin_logprobs"):
                n_missing += 1
            continue
        winner = next((c for c in cands if c.get("won")), None)
        if winner is None:
            continue
        lp, sel = winner.get("fillin_logprob"), winner.get("fillin_prob")
        jw = winner.get("judge_score") if winner.get("judge_kind") in ("score_probs", "token_probs") else None
        if lp is not None:
            logp += lp
        if sel:
            log_sel += math.log(sel)
        if jw:
            log_judge += math.log(jw)
            n_judge += 1
        log_unif += math.log(1.0 / len(cands))
        rows.append((r.get("step_number"), len(cands), lp, sel, jw))

    def _exp(x):
        return math.exp(x) if x > -700 else 0.0

    if not rows:
        # An empty product is 1.0, which would read as a certain path. Say "unknown" instead.
        return rows, {
            "steps_sampled": 0, "steps_without_candidates": n_missing,
            "log_p_fillins": None, "p_fillins": None,
            "log_selection_weight": None, "selection_weight": None,
            "log_uniform_baseline": None, "uniform_baseline": None,
            "log_judge_weight": None, "judge_weight": None, "judge_steps": 0,
        }

    totals = {
        "steps_sampled": len(rows),
        "steps_without_candidates": n_missing,
        "log_p_fillins": logp,
        "p_fillins": _exp(logp),
        "log_selection_weight": log_sel,
        "selection_weight": _exp(log_sel),
        "log_uniform_baseline": log_unif,
        "uniform_baseline": _exp(log_unif),
        "log_judge_weight": log_judge if n_judge else None,
        "judge_weight": _exp(log_judge) if n_judge else None,
        "judge_steps": n_judge,
    }
    return rows, totals


def _path_html(rows, totals, show_steps=True):
    parts = []
    if show_steps and rows:
        hdr = ["step", "cands", "logprob", "P(fill-in)", "sel. weight", "judge weight"]
        head = "".join(f"<th style='padding:2px 8px;text-align:left'>{h}</th>" for h in hdr)
        body = "".join(
            f"<tr><td style='padding:2px 8px'>{st}</td><td style='padding:2px 8px'>{k}</td>"
            f"<td style='padding:2px 8px'>{'&ndash;' if lp is None else format(lp, '+.4f')}</td>"
            f"<td style='padding:2px 8px'>{'&ndash;' if lp is None else format(math.exp(lp), '.4g')}</td>"
            f"<td style='padding:2px 8px'>{'&ndash;' if sel is None else format(sel, '.4f')}</td>"
            f"<td style='padding:2px 8px'>{'&ndash;' if jw is None else format(jw, '.4f')}</td></tr>"
            for st, k, lp, sel, jw in rows
        )
        parts.append(f"<table style='border-collapse:collapse;font-size:11px;margin-top:6px'>"
                     f"<tr style='color:#666'>{head}</tr>{body}</table>")

    if not rows:
        note = (f"{totals['steps_without_candidates']} step(s) carry fill-in probabilities but no "
                "per-candidate record, so the winner cannot be identified &mdash; only runs started "
                "after per-candidate recording was added can be multiplied."
                if totals["steps_without_candidates"] else "No sampled steps in this trajectory.")
        return f"<div style='font-size:12px;margin-top:10px;color:#666'>{note}</div>"

    lines = [
        f"sampled steps: <b>{totals['steps_sampled']}</b>"
        + (f" (+{totals['steps_without_candidates']} with probabilities but no candidate record)"
           if totals["steps_without_candidates"] else ""),
        f"P(fill-ins along path) = <b>{totals['p_fillins']:.6g}</b>"
        f" &nbsp; log = <b>{totals['log_p_fillins']:+.4f}</b>",
        f"selection weight = <b>{totals['selection_weight']:.6g}</b>"
        f" &nbsp; log = <b>{totals['log_selection_weight']:+.4f}</b>"
        f" &nbsp; <span style='color:#666'>uniform baseline"
        f" {totals['uniform_baseline']:.6g}</span>",
    ]
    if totals["judge_steps"]:
        lines.append(f"judge weight = <b>{totals['judge_weight']:.6g}</b>"
                     f" &nbsp; log = <b>{totals['log_judge_weight']:+.4f}</b>"
                     f" &nbsp; <span style='color:#666'>over {totals['judge_steps']} step(s)</span>")
    else:
        lines.append("<span style='color:#666'>judge weight unavailable &mdash; this run used the "
                     "0-10 <code>scores</code> judge, which is not a probability</span>")
    parts.append("<div style='font-size:12px;margin-top:10px;padding-top:8px;"
                 "border-top:1px solid #ddd'>" + "<br>".join(lines) + "</div>")
    return "".join(parts)


def path_probability(source, question=None, show_steps=True):
    """Path probability on its own, without the full trajectory. Returns the totals dict."""
    rows, totals = _path_totals(step_records(source))
    head = f"<div style='font-size:12px'><b>Q</b> {html.escape(str(question))}</div>" if question else ""
    display(HTML(f"<div style='font-family:ui-monospace,monospace'>{head}"
                 f"{_path_html(rows, totals, show_steps)}</div>"))
    return totals


In [50]:
# Run one GAIA question. Set verbosity to DEBUG in cell 4 if you also want the live stream.
from common_setup import question_scorer

example = eval_ds.to_list()[QUESTION_INDEX]
assert not example["file_name"], f"q{QUESTION_INDEX} has an attachment; preprocessing is skipped here"

question, true_answer = example["question"], example["true_answer"]
print(f"Q{QUESTION_INDEX}  task_id={example.get('task_id')}")
print(f"question : {question}")
print(f"true     : {true_answer}")
print("=" * 100)

prediction = agent.run(question)

print("=" * 100)
print(f"prediction : {prediction!r}")
print(f"correct    : {question_scorer(str(prediction), true_answer)}")


Q4  task_id=e1fc63a2-da7a-432f-be78-7c4a95598703
question : If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many thousand hours would it take him to run the distance between the Earth and the Moon its closest approach? Please use the minimum perigee value on the Wikipedia page for the Moon when carrying out your calculation. Round your result to the nearest 1000 hours and do not use any comma separators if necessary.
true     : 17


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many thousand hours would it │
│ take him to run the distance between the Earth and the Moon its closest approach? Please use the minimum        │
│ perigee value on the Wikipedia page for the Moon when carrying out your calculation. Round your result to the   │
│ nearest 1000 hours and do not use any comma separators if necessary.                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2 memory messages


Thought: ──────────────────────────────────────────────────────────────────────────────────────────────────────────
Thought: I need the Moon’s minimum perigee from Wikipedia and Kipchoge’s marathon pace record, then compute the    
time to cover that Earth-Moon distance at that pace. I’ll search for the relevant values, then do the arithmetic   
and round to the nearest 1000 hours.                                                                               

...

[Tracelet] no tool-call arguments to sample this step -- direct execution (no fill-in sampling, no judge).

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  perigee_km = 356400                                                                                              
  pace_min_per_mile = 4 + 33/60                                                                                    
  mile_km = 1.609344                                                                                               
  pace_hours_per_km = (pace_min_per_mile / 60) / mile_km                                                           
  hours = perigee_km * pace_hours_per_km                                                                           
  rounded = round(hours / 1000) * 1000                                                                             
  final_answer(rounded / 1000)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 17.0

[Step 3: Duration 1.56 seconds| Input tokens: 42,327 | Output tokens: 612]

prediction : 17.0
correct    : True


In [53]:
# raw=True swaps the probability bars for the model's own logprobs and their exp().
# The returned dict holds the path products, also printed at the bottom of the render.
path = render_trajectory(agent, question=question, true_answer=true_answer,
                        prediction=prediction, raw=True)
path


{'steps_sampled': 2,
 'steps_without_candidates': 0,
 'log_p_fillins': -9.365127563476562,
 'p_fillins': 8.565974493063945e-05,
 'log_selection_weight': -2.233478459445759,
 'selection_weight': 0.10715504662848041,
 'log_uniform_baseline': -2.1972245773362196,
 'uniform_baseline': 0.11111111111111109,
 'log_judge_weight': -2.097141118779237,
 'judge_weight': 0.12280701754385966,
 'judge_steps': 2}

In [52]:
# Read a trajectory straight out of any completed sweep -- no model calls.
import pickle
from pathlib import Path

RUN_DIR = Path("tracelet_direct_tp_v6n3_react_gpt-4o")   # any run directory
PKL_INDEX = 0

pkl = RUN_DIR / f"{PKL_INDEX}.pkl"
with open(pkl, "rb") as f:
    row = pickle.load(f)
print(f"{pkl}  |  {row['num_steps']} steps  |  correct={row['is_correct']}  |  "
      f"{row['token_counts']['total_tokens']:,} tokens  |  error={row.get('error')}")

path = render_trajectory(pkl, question=row["question"], true_answer=row["true_answer"],
                        prediction=row["prediction"])
path


tracelet_direct_tp_v6n3_react_gpt-4o/0.pkl  |  6 steps  |  correct=False  |  122,913 tokens  |  error=None


{'steps_sampled': 0,
 'steps_without_candidates': 4,
 'log_p_fillins': None,
 'p_fillins': None,
 'log_selection_weight': None,
 'selection_weight': None,
 'log_uniform_baseline': None,
 'uniform_baseline': None,
 'log_judge_weight': None,
 'judge_weight': None,
 'judge_steps': 0}